# KeelNet Stage 1 Experiment Template

Use this notebook as the Stage 1 template for the official team workflow:

1. edit locally in VS Code
2. connect this notebook to a Colab kernel
3. rerun the setup cell after pushing code changes
4. save artifacts to Google Drive

This notebook runs the Stage 1 A/B experiment from `../README.md`:

- Run A: answer-only baseline
- Run B: abstain-aware model
- same backbone, same preprocessing, same training budget
- compare unsupported-answer rate, answerable EM/F1, and abstain metrics


## 1. Setup And Sync

Run this cell first. It mounts Drive, loads `HF_TOKEN` from Colab Secrets if available, clones or updates `/content/KeelNet`, checks out the Stage 1 branch, and installs the project.

Path reminder:

- code runs from `/content/KeelNet`
- team artifacts should save to `/content/drive/Shareddrives/...`
- use `/content/drive/MyDrive/...` only as a personal fallback
- if `DRIVE_PROJECT_DIR` does not start with `/content/drive/`, the path is wrong

Important:

- edit code locally in VS Code
- push code to GitHub
- rerun this cell before training so the Colab runtime updates `/content/KeelNet`


In [1]:
from pathlib import Path
import os
import subprocess
import sys


# Collaboration default: use a shared drive for team-visible artifacts.
USE_SHARED_DRIVE = True
TEAM_DRIVE_NAME = "YourTeamDrive"  # change this if USE_SHARED_DRIVE is True

if USE_SHARED_DRIVE:
    DRIVE_PROJECT_DIR = Path("/content/drive/Shareddrives") / TEAM_DRIVE_NAME / "KeelNet"
else:
    DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/KeelNet")

GIT_REPO_URL = "https://github.com/naufalkmd/KeelNet.git"
GIT_BRANCH = "stage/01-grounded-abstention-baseline"
LOCAL_REPO_DIR = Path("/content/KeelNet")


def run_setup(cmd, *, cwd: Path | None = None) -> None:
    rendered = [str(part) for part in cmd]
    print("$", " ".join(rendered))
    subprocess.run(rendered, check=True, cwd=str(cwd) if cwd else None)


def mount_drive() -> None:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    if not str(DRIVE_PROJECT_DIR).startswith("/content/drive/"):
        raise ValueError(
            f"DRIVE_PROJECT_DIR must point inside /content/drive, got: {DRIVE_PROJECT_DIR}"
        )
    if USE_SHARED_DRIVE and TEAM_DRIVE_NAME == "YourTeamDrive":
        raise ValueError(
            "Set TEAM_DRIVE_NAME to your actual shared drive name, or set USE_SHARED_DRIVE = False."
        )
    DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Drive project dir: {DRIVE_PROJECT_DIR}")


def configure_hf_token() -> None:
    if os.environ.get("HF_TOKEN"):
        print("HF_TOKEN already set in the environment.")
        return

    try:
        from google.colab import userdata
    except ImportError:
        print("Colab secrets are unavailable; continuing without HF_TOKEN.")
        return

    try:
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None

    if token:
        os.environ["HF_TOKEN"] = token
        print("Loaded HF_TOKEN from Colab secrets.")
    else:
        print("HF_TOKEN not found in Colab secrets; continuing with anonymous HF access.")


def ensure_repo() -> Path:
    if (LOCAL_REPO_DIR / ".git").exists():
        run_setup(["git", "fetch", "origin"], cwd=LOCAL_REPO_DIR)
    else:
        run_setup(["git", "clone", GIT_REPO_URL, str(LOCAL_REPO_DIR)])

    run_setup(["git", "checkout", GIT_BRANCH], cwd=LOCAL_REPO_DIR)
    run_setup(["git", "pull", "--ff-only", "origin", GIT_BRANCH], cwd=LOCAL_REPO_DIR)
    return LOCAL_REPO_DIR


mount_drive()
configure_hf_token()
REPO_DIR = ensure_repo().resolve()
os.chdir(REPO_DIR)
print(f"Runtime repo dir: {REPO_DIR}")
print(f"Using repo dir: {REPO_DIR}")
CURRENT_BRANCH = subprocess.run(
    ["git", "rev-parse", "--abbrev-ref", "HEAD"],
    check=True,
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
).stdout.strip()
print(f"Git branch: {CURRENT_BRANCH}")
run_setup([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive project dir: /content/drive/MyDrive/KeelNet
HF_TOKEN not found in Colab secrets; continuing with anonymous HF access.
$ git fetch origin
$ git checkout stage/01-grounded-abstention-baseline
$ git pull --ff-only origin stage/01-grounded-abstention-baseline
Using repo dir: /content/KeelNet
Git branch: stage/01-grounded-abstention-baseline
$ /usr/bin/python3 -m pip install -q -e /content/KeelNet


## 2. Configure The Run

Set your unique `RUN_NAME`, keep the default Stage 1 model unless you are intentionally changing it, and use the smoke-test options before a full run.

This cell also prints the important values you should check before training.

It creates a small `RUN_STARTED.txt` file in the Drive run folder immediately, so you can confirm the Drive path is correct before training finishes.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import torch

REPO_DIR = Path(REPO_DIR).resolve()

# Change this for every teammate and every meaningful run.
RUN_NAME = "yourname-stage1-run"
ARTIFACTS_ROOT = DRIVE_PROJECT_DIR / "artifacts" / "stage1_colab"
OUTPUT_ROOT = ARTIFACTS_ROOT / RUN_NAME
BASELINE_DIR = OUTPUT_ROOT / "baseline"
ABSTAIN_DIR = OUTPUT_ROOT / "abstain"
BASELINE_EVAL = OUTPUT_ROOT / "baseline_eval.json"
ABSTAIN_EVAL = OUTPUT_ROOT / "abstain_eval.json"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_MARKER_FILE = OUTPUT_ROOT / "RUN_STARTED.txt"
RUN_MARKER_FILE.write_text(
    "\n".join(
        [
            f"run_name={RUN_NAME}",
            f"repo_dir={REPO_DIR}",
            "status=configured",
            "note=This file is created when the config cell runs.",
        ]
    )
    + "\n",
    encoding="utf-8",
)

# Keep these fixed across baseline and abstain runs for a fair comparison.
MODEL_NAME = "distilbert-base-uncased"
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8

# Set these to small values for a quick smoke test before the full run.
MAX_TRAIN_SAMPLES = None
MAX_EVAL_SAMPLES = None

# Optional quick validation step before full training.
RUN_SMOKE_TEST = False
SMOKE_TEST_TRAIN_SAMPLES = 32
SMOKE_TEST_EVAL_SAMPLES = 32
SMOKE_TEST_EPOCHS = 1

print(f"Repo dir: {REPO_DIR}")
print(f"Artifacts root: {ARTIFACTS_ROOT}")
print(f"Run output dir: {OUTPUT_ROOT}")
print(f"Run marker file: {RUN_MARKER_FILE}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def run(cmd):
    print("$", " ".join(str(part) for part in cmd))
    completed = subprocess.run([str(part) for part in cmd], cwd=REPO_DIR, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()


## 3. Validate The Environment

Run the project tests before training. This confirms the installed code path is at least minimally healthy inside the Colab runtime.


In [ ]:
run([sys.executable, "-m", "unittest", "discover", "-s", str(REPO_DIR / "tests")])


## 4. Optional Smoke Test

This cell runs a tiny baseline-only training command. Use it to catch path, dependency, and runtime issues before the full experiment.

Set `RUN_SMOKE_TEST = False` in the config cell if you want to skip it after the environment is already trusted.


In [ ]:
if RUN_SMOKE_TEST:
    smoke_dir = OUTPUT_ROOT / "smoke-baseline"
    run([
        sys.executable,
        "-m",
        "keelnet.train",
        "--mode",
        "baseline",
        "--output-dir",
        smoke_dir,
        "--model-name",
        MODEL_NAME,
        "--num-train-epochs",
        str(SMOKE_TEST_EPOCHS),
        "--train-batch-size",
        "2",
        "--eval-batch-size",
        "2",
        "--max-train-samples",
        str(SMOKE_TEST_TRAIN_SAMPLES),
        "--max-eval-samples",
        str(SMOKE_TEST_EVAL_SAMPLES),
    ])
else:
    print("Smoke test skipped. Set RUN_SMOKE_TEST = True in the config cell to run it.")


In [ ]:
print("DRIVE_PROJECT_DIR =", DRIVE_PROJECT_DIR)
print("ARTIFACTS_ROOT =", ARTIFACTS_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("exists DRIVE_PROJECT_DIR:", DRIVE_PROJECT_DIR.exists())
print("exists OUTPUT_ROOT:", OUTPUT_ROOT.exists())

for p in sorted(DRIVE_PROJECT_DIR.rglob("*")):
    print(p)


## 5. Train The Two Stage 1 Runs

This cell trains the answer-only baseline and then the abstain-aware model. Keep the shared settings fixed so the comparison stays fair.


In [ ]:
def train_run(mode: str, output_dir: Path):
    cmd = [
        sys.executable,
        "-m",
        "keelnet.train",
        "--mode",
        mode,
        "--output-dir",
        str(output_dir),
        "--model-name",
        MODEL_NAME,
        "--num-train-epochs",
        str(NUM_EPOCHS),
        "--train-batch-size",
        str(TRAIN_BATCH_SIZE),
        "--eval-batch-size",
        str(EVAL_BATCH_SIZE),
    ]
    if MAX_TRAIN_SAMPLES is not None:
        cmd.extend(["--max-train-samples", str(MAX_TRAIN_SAMPLES)])
    if MAX_EVAL_SAMPLES is not None:
        cmd.extend(["--max-eval-samples", str(MAX_EVAL_SAMPLES)])
    run(cmd)


train_run("baseline", BASELINE_DIR)
train_run("abstain", ABSTAIN_DIR)

## 6. Evaluate Both Runs

This cell evaluates both trained models and writes the result JSON files into your Drive run folder.


In [ ]:
def evaluate_run(mode: str, model_dir: Path, output_path: Path):
    cmd = [
        sys.executable,
        "-m",
        "keelnet.evaluate",
        "--mode",
        mode,
        "--model-path",
        str(model_dir),
        "--output-path",
        str(output_path),
        "--eval-batch-size",
        str(EVAL_BATCH_SIZE),
    ]
    if MAX_EVAL_SAMPLES is not None:
        cmd.extend(["--max-eval-samples", str(MAX_EVAL_SAMPLES)])
    run(cmd)


evaluate_run("baseline", BASELINE_DIR, BASELINE_EVAL)
evaluate_run("abstain", ABSTAIN_DIR, ABSTAIN_EVAL)


## 7. Compare The Main Metrics

This cell loads the saved evaluation JSON files and builds the main comparison table for Stage 1.


In [ ]:
baseline_results = json.loads(BASELINE_EVAL.read_text())
abstain_results = json.loads(ABSTAIN_EVAL.read_text())

comparison = pd.DataFrame(
    [
        {"model": "Run A", **baseline_results["dev_metrics"]},
        {"model": "Run B", **abstain_results["dev_metrics"]},
    ]
)
comparison = comparison[
    [
        "model",
        "answerable_em",
        "answerable_f1",
        "unsupported_answer_rate",
        "abstain_precision",
        "abstain_recall",
        "abstain_f1",
        "overall_em",
        "overall_f1",
    ]
]
comparison.round(2)


## 8. Inspect The Abstain Threshold

This cell shows the validation-selected threshold and validation metrics for the abstain-aware run.


In [ ]:
print("Selected abstain threshold:", abstain_results["selected_threshold"])
print("\nValidation metrics for Run B:")
pd.Series(abstain_results["validation_metrics"]).round(2)


## What To Check Next

A win is not just a lower unsupported-answer rate.

Check all three:

- unsupported-answer rate goes down
- answerable F1 does not collapse
- the abstain-aware model is not simply over-abstaining

After that, inspect failure cases using the saved `*_eval.json` files under the printed `OUTPUT_ROOT` directory in Drive.
